Step 1: Import lib

In [1]:
import argparse
import json
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm import tqdm

Step 2: Config

In [31]:
# OPTIMIZE
DATASET_ROOT = r"D:\dataset"
MODEL_NAME = "resnext"
EPOCHS = 1

BATCH_SIZE = 64          # thử 64 trước, nếu OOM thì xuống 32
IMG_SIZE = 224
LR = 0.01
MOMENTUM = 0.9
TRAIN_RATIO = 0.7
SEED = 42

NUM_WORKERS = 8          # nếu Windows không ổn thì hạ 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

USE_AMP = True
USE_COMPILE = False      # bật sau khi baseline ổn
USE_CHANNELS_LAST = True

# Chỉ để run thử nhanh, rất nên có:
MAX_IMAGES_PER_CLASS = None   # ví dụ 20000 nếu muốn test nhanh hơn nhiều

PRETRAINED = False
OUTPUT_DIR = Path("./outputs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
# DATASET_ROOT = "/Users/tranminhhuy/Documents/dataset" # update dataset path
DATASET_ROOT = r"D:\dataset"
MODEL_NAME = "resnext"   # "resnext", "wideresnet", "efficientnet"
EPOCHS = 1 # look back train set
# BATCH_SIZE = 8
BATCH_SIZE = 32
IMG_SIZE = 224
LR = 0.01 # learning rate -> based on original paper
MOMENTUM = 0.9 # based on original paper
TRAIN_RATIO = 0.7 # percentage dataset used to train
SEED = 42 # for random
# NUM_WORKERS = 0 # worker helps to load data
NUM_WORKERS = 4 # worker helps to load data
PRETRAINED = False # model does not use data learnt before

PIN_MEMORY = True # help tranfer effeciently  data from RAM to GPU when using CUDA
PERSISTENT_WORKERS = True # when NUM_WORKERS > 0

OUTPUT_DIR = Path("./outputs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Implement view RAM, VRAM, etc
HOW TO USE: just call print_system_stats()

In [32]:
import os
import time
import psutil
import shutil
import torch

def bytes_to_gb(x):
    return x / (1024 ** 3)

def print_system_stats():
    print("===== SYSTEM STATS =====")
    
    # RAM
    vm = psutil.virtual_memory()
    print(f"RAM total     : {bytes_to_gb(vm.total):.2f} GB")
    print(f"RAM used      : {bytes_to_gb(vm.used):.2f} GB")
    print(f"RAM available : {bytes_to_gb(vm.available):.2f} GB")
    print(f"RAM percent   : {vm.percent}%")
    
    # Disk
    disk = shutil.disk_usage(DATASET_ROOT if os.path.exists(DATASET_ROOT) else ".")
    print(f"Disk total    : {bytes_to_gb(disk.total):.2f} GB")
    print(f"Disk used     : {bytes_to_gb(disk.used):.2f} GB")
    print(f"Disk free     : {bytes_to_gb(disk.free):.2f} GB")

    # GPU
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        total_vram = props.total_memory
        reserved = torch.cuda.memory_reserved(0)
        allocated = torch.cuda.memory_allocated(0)
        free_est = total_vram - reserved

        print(f"GPU name      : {torch.cuda.get_device_name(0)}")
        print(f"VRAM total    : {bytes_to_gb(total_vram):.2f} GB")
        print(f"VRAM reserved : {bytes_to_gb(reserved):.2f} GB")
        print(f"VRAM allocated: {bytes_to_gb(allocated):.2f} GB")
        print(f"VRAM free est.: {bytes_to_gb(free_est):.2f} GB")
    else:
        print("GPU           : CUDA not available")

    print("========================")

Step 3: Seed & device #local mac

In [20]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()
print("Using device:", device)
print_system_stats()

Using device: cuda
===== SYSTEM STATS =====
RAM total     : 31.81 GB
RAM used      : 14.01 GB
RAM available : 17.80 GB
RAM percent   : 44.0%
Disk total    : 931.51 GB
Disk used     : 493.72 GB
Disk free     : 437.79 GB
GPU name      : NVIDIA GeForce RTX 3060
VRAM total    : 12.00 GB
VRAM reserved : 0.00 GB
VRAM allocated: 0.00 GB
VRAM free est.: 12.00 GB


Step 3: Seed & device local win

In [33]:
import torch
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()

print("Using device:", device)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Using device: cuda
CUDA available: True
GPU name: NVIDIA GeForce RTX 3060


Step 4: Transforms

In [22]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

Step 5: Load dataset

In [34]:
base_dataset = datasets.ImageFolder(DATASET_ROOT)
train_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=train_transform)
test_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=test_transform)

print("Classes:", base_dataset.classes)
print("Num classes:", len(base_dataset.classes))
print("Total images:", len(base_dataset))
print_system_stats()

Classes: ['aerial', 'architecture', 'event', 'fashion', 'food', 'nature', 'sports', 'street', 'wedding']
Num classes: 9
Total images: 890053
===== SYSTEM STATS =====
RAM total     : 31.81 GB
RAM used      : 19.08 GB
RAM available : 12.73 GB
RAM percent   : 60.0%
Disk total    : 931.51 GB
Disk used     : 493.72 GB
Disk free     : 437.79 GB
GPU name      : NVIDIA GeForce RTX 3060
VRAM total    : 12.00 GB
VRAM reserved : 4.15 GB
VRAM allocated: 0.30 GB
VRAM free est.: 7.85 GB


Step 6: Split dataset with 70% train and 30% test

In [35]:
from collections import defaultdict
import random

def stratified_split_indices(targets, train_ratio=0.7, seed=42, max_images_per_class=None):
    rng = random.Random(seed)
    class_to_indices = defaultdict(list)

    for idx, label in enumerate(targets):
        class_to_indices[label].append(idx)

    train_indices = []
    test_indices = []

    for label, indices in class_to_indices.items():
        indices = indices[:]
        rng.shuffle(indices)

        if max_images_per_class is not None:
            indices = indices[:max_images_per_class]

        split_idx = int(len(indices) * train_ratio)
        train_indices.extend(indices[:split_idx])
        test_indices.extend(indices[split_idx:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return train_indices, test_indices


train_indices, test_indices = stratified_split_indices(
    base_dataset.targets,
    train_ratio=TRAIN_RATIO,
    seed=SEED,
    max_images_per_class=MAX_IMAGES_PER_CLASS
)

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Train samples: 623033
Test samples: 267020


Step 7: Dataloader prepare data so that model can read it in batches

In [38]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
)

print("Train loader batches:", len(train_loader))
print("Test loader batches:", len(test_loader))

Train loader batches: 9735
Test loader batches: 4173


Step 8: Build model: create model based on the config above

In [39]:
# Step 8: Build model NEW OPTIMIZE

# Safe defaults in case these are not added yet in Config
USE_CHANNELS_LAST = globals().get("USE_CHANNELS_LAST", True)
USE_COMPILE = globals().get("USE_COMPILE", False)

def build_model(model_name: str, num_classes: int, pretrained: bool = False):
    model_name = model_name.lower()

    if model_name == "resnext":
        weights = models.ResNeXt50_32X4D_Weights.DEFAULT if pretrained else None
        model = models.resnext50_32x4d(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unsupported model: {model_name}")

# CUDA tuning
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

model = build_model(
    model_name=MODEL_NAME,
    num_classes=len(base_dataset.classes),
    pretrained=PRETRAINED
)

# Optional memory format optimization for CNNs on CUDA
if device.type == "cuda" and USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)

model = model.to(device)

# Optional compile for extra speed after baseline is stable
if device.type == "cuda" and USE_COMPILE:
    model = torch.compile(model)

criterion = nn.CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=LR, momentum=MOMENTUM)

print("Model name:", MODEL_NAME)
print("Num classes:", len(base_dataset.classes))
print("Device:", device)
print("Pretrained:", PRETRAINED)
print("Channels last:", USE_CHANNELS_LAST if device.type == "cuda" else False)
print("Use compile:", USE_COMPILE if device.type == "cuda" else False)

Model name: resnext
Num classes: 9
Device: cuda
Pretrained: False
Channels last: True
Use compile: False


In [ ]:
def build_model(model_name: str, num_classes: int, pretrained: bool = False):
    model_name = model_name.lower()

    if model_name == "resnext":
        weights = models.ResNeXt50_32X4D_Weights.DEFAULT if pretrained else None
        model = models.resnext50_32x4d(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unsupported model: {model_name}")


model = build_model(
    model_name=MODEL_NAME,
    num_classes=len(base_dataset.classes),
    pretrained=PRETRAINED
).to(device)

criterion = nn.CrossEntropyLoss() # loss function
optimizer = SGD(model.parameters(), lr=LR, momentum=MOMENTUM) # optimizer

print("Model name:", MODEL_NAME)
print("Num classes:", len(base_dataset.classes))
print("Device:", device)

Model name: resnext
Num classes: 9
Device: cuda


Step 9: Create Train and evaluate function

In [40]:
# Step 9: Train / evaluate functions optimize

# Safe defaults in case these are not added yet in Config
USE_AMP = globals().get("USE_AMP", True)
USE_CHANNELS_LAST = globals().get("USE_CHANNELS_LAST", True)

use_amp = USE_AMP and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if device.type == "cuda" else None

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        if device.type == "cuda":
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
        else:
            images = images.to(device)
            labels = labels.to(device)

        if device.type == "cuda" and USE_CHANNELS_LAST:
            images = images.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        if device.type == "cuda":
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        if device.type == "cuda":
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
        else:
            images = images.to(device)
            labels = labels.to(device)

        if device.type == "cuda" and USE_CHANNELS_LAST:
            images = images.contiguous(memory_format=torch.channels_last)

        if device.type == "cuda":
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


print("AMP enabled:", use_amp)
print("Channels last:", USE_CHANNELS_LAST if device.type == "cuda" else False)

AMP enabled: True
Channels last: True


In [28]:
def train_one_epoch(model, loader, criterion, optimizer, device): #train_loader teach model
    model.train()
    running_loss = 0.0
    correct = 0 # the number of right prediction
    total = 0 # total images go through model

    for images, labels in tqdm(loader, desc="Training", leave=False): 
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device): #test_loader check model after learning
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

Step 10: Start train model 

In [ ]:
# Step 10: Start train model optimize

history = []
best_acc = 0.0

train_start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device
    )

    epoch_time = time.perf_counter() - epoch_start

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print(f"Epoch time: {epoch_time:.2f} seconds")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "epoch_time_sec": epoch_time,
    })

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "classes": base_dataset.classes,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE,
                "train_ratio": TRAIN_RATIO,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "momentum": MOMENTUM,
                "epochs": EPOCHS,
                "pretrained": PRETRAINED,
            },
            OUTPUT_DIR / f"best_{MODEL_NAME}.pth"
        )
        print(f"Saved best model to: {OUTPUT_DIR / f'best_{MODEL_NAME}.pth'}")

total_train_time = time.perf_counter() - train_start_time

print("\nBest test accuracy:", best_acc)
print(f"Total training time: {total_train_time:.2f} seconds")


Epoch 1/1


Training:   0%|          | 45/9735 [02:36<6:32:30,  2.43s/it] 

In [ ]:
history = []
best_acc = 0.0

train_start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    epoch_time = time.perf_counter() - epoch_start

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {tsrain_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print(f"Epoch time: {epoch_time:.2f} seconds")

    history.append({
        "epoch": epoch,s
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "epoch_time_sec": epoch_time,
    })

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "classes": base_dataset.classes,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE,
            },
            OUTPUT_DIR / f"best_{MODEL_NAME}.pth"
        )

total_train_time = time.perf_counter() - train_start_time

print("\nBest test accuracy:", best_acc)
print(f"Total training time: {total_train_time:.2f} seconds")


Epoch 1/1


KeyboardInterrupt: 